#### 07. Training and tuning

Logistic regression and random forest from `task2/code/models.py`, trained on the training table named by `RAIN_TRAIN` (by default the subsampled one: 2017–2022, dry rows weighted back up with `w`) and tuned on 2023 by RPS. Both output the 9 class probabilities; the threshold mapper turns them into P(Y > θ) for any θ. The test year 2024 is not used here.

**Setup**

In [ ]:
%load_ext autoreload
%autoreload 2
import time
import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F

import config as C
import evaluate as E
import mapper as M
import models as MD

spark = C.get_spark("task2-training")

TRAIN_NAME = C.TRAIN_NAME
train = spark.read.parquet(str(C.ASSEMBLED / TRAIN_NAME)).cache()
val = spark.read.parquet(str(C.ASSEMBLED / "val")).cache()
TIERS = tuple(t for t in ("t0", "t1", "t2", "t3") if any(c.startswith(f"{t}_") for c in train.columns))
FULL = "+".join(t.upper() for t in TIERS)
ALL = MD.feature_columns(train, TIERS)
print(len(ALL), "features in", FULL)

41 features


**Threshold mapper**

The mapper stores the rainfall values inside each wet class of the training table, so P(Y > θ) can be read off for any θ. It is part of the trained model.

In [2]:
mp = M.fit(train)
M.save(mp, C.MODELS / "mapper.json")
{k: (len(v["y"]), max(v["y"])) for k, v in mp["classes"].items()}

{1: (1, 0.2),
 2: (1, 0.4),
 3: (3, 1.0),
 4: (5, 2.0),
 5: (27, 5.0),
 6: (50, 10.0),
 7: (100, 20.0),
 8: (392, 71.8)}

**Tuning on 2023**

Both models use every feature of the training table: 41 with Tiers 0 to 2, 58 with Tier 3.

Logistic regression is fitted to convergence (at most 1,000 iterations), once on the features as they are and three times with non-linear inputs added: the main rain, wetness, distance and time-since-rain features cut into bands, and an hour-of-day × season interaction. The strength of the L2 penalty is tuned for the non-linear version. 

The random forest is screened one setting at a time (tree depth, minimum rows per leaf, features tried per split, number of bins, split criterion), keeping the value with the lowest RPS on 2023 before moving to the next. Every screened forest has RAIN_RF_TREES trees; fit time and tree size are recorded to choose the setting for the full training table.

In [ ]:
def val_rps(model):
    return E.brier_rps(E.add_exceedance(model.transform(val)))[1]

In [ ]:
LR_SPECS = [{"reg_param": 1e-5, "nonlinear": False},
            {"reg_param": 1e-6, "nonlinear": True},
            {"reg_param": 1e-5, "nonlinear": True},
            {"reg_param": 1e-4, "nonlinear": True}]
lr_fits, lr_rows = [], []
for spec in LR_SPECS:
    start = time.time()
    model = MD.build_lr(ALL, **spec).fit(train)
    lr_fits.append(model)
    lr_rows.append({**spec, "val RPS": val_rps(model), "iterations": model.stages[-1].summary.totalIterations,
                    "fit min": round((time.time() - start) / 60, 1)})
lr_table = pd.DataFrame(lr_rows)
lr_table.to_csv(C.RESULTS / "lr_tuning.csv", index=False)
LR_BEST = int(lr_table["val RPS"].idxmin())
lr_spec = LR_SPECS[LR_BEST]
lr_table.round({"val RPS": 6})

regParam
0.00001    0.096338
0.00010    0.096387
0.00100    0.096695
Name: val RPS, dtype: float64

In [ ]:
RF_GRID = {"max_depth": [14, 17, 20, 24],
           "min_instances": [20, 50, 100, 300],
           "feature_subset": ["sqrt", "onethird", "0.5"],
           "max_bins": [64, 256],
           "impurity": ["gini", "entropy"]}
SWEEPS = 1
rf_spec = {"max_depth": 14, "min_instances": 100, "feature_subset": "sqrt", "max_bins": 64, "impurity": "gini"}
tried = {}


def rf_try(spec):
    key = tuple(spec.items())
    if key not in tried:
        start = time.time()
        model = MD.build_rf(ALL, num_trees=C.RF_TREES, **spec).fit(train)
        minutes = (time.time() - start) / 60
        tried[key] = {**spec, "val RPS": val_rps(model), "fit min": round(minutes, 1),
                      "nodes": model.stages[-1].totalNumNodes}
        pd.DataFrame(tried.values()).to_csv(C.RESULTS / "rf_screening.csv", index=False)
        print(tried[key], flush=True)
    return tried[key]["val RPS"]


for _ in range(SWEEPS):
    for param, values in RF_GRID.items():
        rf_spec[param] = min(values, key=lambda v: rf_try({**rf_spec, param: v}))
rf_spec

maxDepth
10    0.094937
12    0.093472
14    0.092758
Name: val RPS, dtype: float64

In [ ]:
pd.DataFrame(tried.values()).sort_values("val RPS").round({"val RPS": 6})

**Ablation by tier**

With the chosen settings, both models are also trained on each smaller set of tiers (Tier 0, Tiers 0–1 and, when Tier 3 is present, Tiers 0–2), and all of them are scored against the baselines on 2023, overall and split by whether the station is dry or wet now.

In [ ]:
models = {("LR", FULL): lr_fits[LR_BEST],
          ("RF", FULL): MD.build_rf(ALL, num_trees=C.RF_TREES, **rf_spec).fit(train)}
for tiers in [TIERS[:n] for n in range(1, len(TIERS))]:
    feats = MD.feature_columns(train, tiers)
    name = "+".join(t.upper() for t in tiers)
    models[("LR", name)] = MD.build_lr(feats, **lr_spec).fit(train)
    models[("RF", name)] = MD.build_rf(feats, num_trees=C.RF_TREES, **rf_spec).fit(train)

rows = []
for (kind, tiers), model in models.items():
    pred = model.transform(val).select("label", "y_mm", "w", "regime", "probability", "prediction").cache()
    rows.append({"model": f"{kind} {tiers}", "subset": "all", **E.score(pred)})
    for regime in ["dry_now", "wet_now"]:
        rows.append({"model": f"{kind} {tiers}", "subset": regime, **E.score(pred.filter(F.col("regime") == regime))})
    pred.unpersist()

baselines = pd.read_csv(C.RESULTS / "baselines_val.csv").rename(columns={"forecast": "model"})
results = pd.concat([baselines, pd.DataFrame(rows)]).set_index(["subset", "model"]).sort_index()
results.to_csv(C.RESULTS / "models_val.csv")
results.loc["all"].sort_values("RPS").round(4)

,RPS,log loss,Brier 0,Brier 1,Brier 5,Brier 20,AUC-PR 0,AUC-PR 1,AUC-PR 5,AUC-PR 20
model,,,,,,,,,,
RF T0+T1+T2,0.0928,0.1952,0.0258,0.0127,0.0058,0.0010,0.8064,0.6993,0.5068,0.2014
LR T0+T1+T2,0.0963,0.2020,0.0270,0.0132,0.0058,0.0010,0.7908,0.6787,0.4992,0.1844
RF T0+T1,0.1117,0.2385,0.0327,0.0149,0.0063,0.0010,0.7004,0.5858,0.4148,0.1693
LR T0+T1,0.1140,0.2403,0.0332,0.0154,0.0064,0.0010,0.6905,0.5681,0.3844,0.1349
persistence,0.1266,0.2640,0.0360,0.0173,0.0073,0.0011,0.6589,0.4815,0.2600,0.0542
climatology,0.1935,0.3615,0.0607,0.0246,0.0086,0.0011,0.1403,0.0621,0.0227,0.0030
RF T0,0.1944,0.3592,0.0612,0.0246,0.0086,0.0011,0.1308,0.0610,0.0246,0.0039
LR T0,0.1947,0.3585,0.0612,0.0247,0.0086,0.0011,0.1210,0.0539,0.0203,0.0033
constant,0.1977,0.3689,0.0624,0.0249,0.0086,0.0011,0.0668,0.0256,0.0087,0.0011


In [6]:
cols = ["RPS", "log loss", "Brier 0", "AUC-PR 0", "AUC-PR 5"]
pd.concat({"dry now": results.loc["dry_now"][cols], "wet now": results.loc["wet_now"][cols]}, axis=1).sort_values(("dry now", "RPS")).round(4)

dry now                                    wet now           \
                RPS log loss Brier 0 AUC-PR 0 AUC-PR 5     RPS log loss   
model                                                                     
RF T0+T1+T2  0.0538   0.1133  0.0187   0.4137   0.1649  0.6356   1.3358   
LR T0+T1+T2  0.0551   0.1153  0.0192   0.3836   0.1520  0.6709   1.4107   
LR T0+T1     0.0662   0.1468  0.0238   0.1148   0.0083  0.7799   1.5425   
RF T0+T1     0.0664   0.1495  0.0239   0.1228   0.0089  0.7427   1.4763   
persistence  0.0678   0.1623  0.0250   0.0256   0.0028  0.9422   1.6764   
RF T0        0.0700   0.1711  0.0259   0.0513   0.0095  1.9302   2.9822   
LR T0        0.0703   0.1713  0.0260   0.0473   0.0073  1.9300   2.9697   
climatology  0.0705   0.1714  0.0261   0.0538   0.0089  1.9093   3.0117   
constant     0.0706   0.1764  0.0262   0.0256   0.0028  1.9704   3.0534   

                                       
            Brier 0 AUC-PR 0 AUC-PR 5  
model                                  
RF T0+T1+T2  0.1244   0.9432   0.6251  
LR T0+T1+T2  0.1345   0.9328   0.6240  
LR T0+T1     0.1643   0.9025   0.5396  
RF T0+T1     0.1553   0.9133   0.5755  
persistence  0.1880   0.8341   0.3515  
RF T0        0.5522   0.6668   0.1181  
LR T0        0.5520   0.6610   0.1230  
climatology  0.5437   0.6635   0.1152  
constant     0.5681   0.6416   0.0911

**Calibration and feature importance**

In [7]:
best = {"LR": models[("LR", FULL)], "RF": models[("RF", FULL)]}
tables = {}
for kind, model in best.items():
    ex = E.add_exceedance(model.transform(val))
    for th in [0.0, 5.0]:
        tables[f"{kind}, θ = {th:g} mm"] = E.reliability(ex, th).set_index("bin")[["n", "mean_forecast", "observed"]]
pd.concat(tables, axis=1).round(3)

LR, θ = 0 mm                        LR, θ = 5 mm                         \
               n mean_forecast observed            n mean_forecast observed   
bin                                                                           
0      5911493.0         0.010    0.010    6601205.0         0.003    0.003   
1       246238.0         0.141    0.151      44072.0         0.142    0.154   
2       110293.0         0.245    0.277      21319.0         0.246    0.240   
3        71110.0         0.347    0.381      14085.0         0.347    0.332   
4        52868.0         0.448    0.467      10419.0         0.448    0.429   
5        44254.0         0.549    0.554       8177.0         0.548    0.511   
6        42212.0         0.650    0.651       6896.0         0.649    0.603   
7        45649.0         0.751    0.762       5757.0         0.749    0.705   
8        48936.0         0.852    0.838       4769.0         0.849    0.841   
9       146221.0         0.973    0.969       2575.0         0.935    0.937   

    RF, θ = 0 mm                        RF, θ = 5 mm                         
               n mean_forecast observed            n mean_forecast observed  
bin                                                                          
0      5922165.0         0.011    0.010    6583974.0         0.003    0.003  
1       202743.0         0.145    0.144      61419.0         0.141    0.146  
2       119902.0         0.246    0.243      28297.0         0.245    0.260  
3        77209.0         0.347    0.348      14182.0         0.346    0.367  
4        57579.0         0.448    0.450       9830.0         0.447    0.468  
5        50351.0         0.549    0.553       7353.0         0.547    0.590  
6        44605.0         0.650    0.659       5665.0         0.649    0.698  
7        46147.0         0.751    0.754       4361.0         0.748    0.816  
8        52033.0         0.852    0.868       3706.0         0.847    0.912  
9       146540.0         0.968    0.979        487.0         0.913    0.967

In [8]:
importance = pd.Series(best["RF"].stages[-1].featureImportances.toArray(), index=ALL, name="importance")
importance.sort_values(ascending=False).head(15).round(4)

t2_idw_rain_15        0.1873
t2_wet_frac_10km      0.1248
t2_idw_rain_30        0.0927
t1_max5_30            0.0656
t1_rain_15            0.0644
t1_min_since_rain     0.0606
t1_rain_30            0.0514
t2_island_wet_frac    0.0508
t1_rain_5             0.0494
t2_wet_frac_5km       0.0494
t2_dist_rain_km       0.0423
t2_idw_trend          0.0290
t2_max_rain_30        0.0218
t2_island_rain_30     0.0160
t1_wet_slots_60       0.0144
Name: importance, dtype: float64

**Saving the models**

In [9]:
for kind, model in best.items():
    model.write().overwrite().save(str(C.MODELS / kind.lower()))
sorted(p.name for p in C.MODELS.iterdir())

['lr', 'mapper.json', 'rf']